In [1]:
# ============================================================
# UPDATE_JSON.py
# Régénération du fichier JSON du référentiel (sans NaN)
# À exécuter après avoir corrigé le référentiel Excel
# ============================================================

import os
import pandas as pd
import json
from datetime import datetime

# ============================================================
# 1. PARAMÈTRES
# ============================================================

# Chemin du fichier Excel corrigé
CHEMIN_EXCEL = "../metadata/mon_referentiel_final.xlsx"

# Chemin du fichier JSON à générer
CHEMIN_JSON = "../metadata/referentiel.json"

# Version du référentiel
VERSION = "2.1"

# ============================================================
# 2. FONCTIONS
# ============================================================

def clean_nan(df):
    """Remplace les NaN par None pour un JSON propre"""
    return df.where(pd.notnull(df), None)

def charger_onglet(excel, nom_onglet):
    """Charge un onglet avec gestion des erreurs"""
    try:
        df = pd.read_excel(excel, sheet_name=nom_onglet)
        print(f"   ✅ {nom_onglet} : {len(df)} lignes")
        return df
    except Exception as e:
        print(f"   ⚠️ {nom_onglet} : non trouvé ou vide ({e})")
        return pd.DataFrame()

# ============================================================
# 3. CHARGEMENT DES DONNÉES
# ============================================================

print("=" * 60)
print("🚀 RÉGÉNÉRATION DU RÉFÉRENTIEL JSON (SANS NaN)")
print("=" * 60)

# Vérifier que le fichier Excel existe
if not os.path.exists(CHEMIN_EXCEL):
    print(f"\n❌ Fichier Excel non trouvé : {CHEMIN_EXCEL}")
    print("   Vérifiez le chemin et le nom du fichier.")
    exit(1)

print(f"\n📁 Fichier Excel trouvé : {CHEMIN_EXCEL}")

# Charger tous les onglets
print("\n📋 Chargement des onglets...")

df_modele = charger_onglet(CHEMIN_EXCEL, "Modele_donnees")
df_objets = charger_onglet(CHEMIN_EXCEL, "Objets_metier")
df_mappings = charger_onglet(CHEMIN_EXCEL, "Mappings")
df_contraintes = charger_onglet(CHEMIN_EXCEL, "Contraintes")
df_regles = charger_onglet(CHEMIN_EXCEL, "Regles")
df_valeurs = charger_onglet(CHEMIN_EXCEL, "Valeurs_reference")
df_ponderation = charger_onglet(CHEMIN_EXCEL, "Ponderation_Score")
df_mappings_xgboost = charger_onglet(CHEMIN_EXCEL, "Mappings_XGBoost_Features")
df_changelog = charger_onglet(CHEMIN_EXCEL, "Changelog_Mapping")

# ============================================================
# 4. VÉRIFICATION DES RÈGLES (R007 en particulier)
# ============================================================

print("\n🔍 Vérification des règles corrigées...")

if not df_regles.empty:
    for index, row in df_regles.iterrows():
        if row.get('REGLE_ID') == 'R007':
            expression = str(row.get('EXPRESSION_SQL', ''))
            if 'COUNT(*)' in expression:
                print(f"   ✅ R007 corrigé : {expression[:80]}...")
            elif 'COUNT()' in expression:
                print(f"   ❌ R007 toujours incorrect : {expression[:80]}...")
                print("   ⚠️ Veuillez corriger la syntaxe SQL (COUNT() → COUNT(*))")
            else:
                print(f"   ℹ️ R007 : {expression[:80]}...")
else:
    print("   ⚠️ Aucune règle trouvée dans l'onglet Regles")

# ============================================================
# 5. CONSTRUCTION DU JSON
# ============================================================

print("\n📦 Construction du JSON...")

# Nettoyer les NaN
data = {
    "version": VERSION,
    "date_maj": datetime.now().isoformat(),
    "Modele_donnees": clean_nan(df_modele).to_dict(orient='records') if not df_modele.empty else [],
    "Objets_metier": clean_nan(df_objets).to_dict(orient='records') if not df_objets.empty else [],
    "Mappings": clean_nan(df_mappings).to_dict(orient='records') if not df_mappings.empty else [],
    "Contraintes": clean_nan(df_contraintes).to_dict(orient='records') if not df_contraintes.empty else [],
    "Regles": clean_nan(df_regles).to_dict(orient='records') if not df_regles.empty else [],
    "Valeurs_reference": clean_nan(df_valeurs).to_dict(orient='records') if not df_valeurs.empty else [],
    "Ponderation_Score": clean_nan(df_ponderation).to_dict(orient='records') if not df_ponderation.empty else [],
    "Mappings_XGBoost_Features": clean_nan(df_mappings_xgboost).to_dict(orient='records') if not df_mappings_xgboost.empty else [],
    "Changelog_Mapping": clean_nan(df_changelog).to_dict(orient='records') if not df_changelog.empty else []
}

# ============================================================
# 6. SAUVEGARDE DU JSON
# ============================================================

print(f"\n💾 Sauvegarde du JSON : {CHEMIN_JSON}")

try:
    with open(CHEMIN_JSON, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print("   ✅ Sauvegarde réussie")
except Exception as e:
    print(f"   ❌ Erreur lors de la sauvegarde : {e}")
    exit(1)

# ============================================================
# 7. VÉRIFICATION DU JSON
# ============================================================

print("\n🔍 Vérification du JSON généré...")

# Vérifier la présence de NaN
with open(CHEMIN_JSON, 'r', encoding='utf-8') as f:
    contenu = f.read()
    
if 'NaN' in contenu:
    print("   ⚠️ Attention : des NaN sont encore présents dans le JSON")
    print("   Cela peut indiquer un problème de nettoyage")
else:
    print("   ✅ Aucun NaN détecté dans le JSON")

# Afficher la taille du fichier
taille = os.path.getsize(CHEMIN_JSON)
print(f"   📊 Taille du fichier : {taille} octets")

# ============================================================
# 8. AFFICHAGE DU RÉSUMÉ
# ============================================================

print("\n" + "=" * 60)
print("📊 RÉSUMÉ DU RÉFÉRENTIEL")
print("=" * 60)

print(f"   Version : {data['version']}")
print(f"   Date de mise à jour : {data['date_maj']}")
print(f"\n   Nombre d'entités :")
print(f"   - Modele_donnees : {len(data['Modele_donnees'])}")
print(f"   - Objets_metier : {len(data['Objets_metier'])}")
print(f"   - Mappings : {len(data['Mappings'])}")
print(f"   - Contraintes : {len(data['Contraintes'])}")
print(f"   - Regles : {len(data['Regles'])}")
print(f"   - Valeurs_reference : {len(data['Valeurs_reference'])}")
print(f"   - Ponderation_Score : {len(data['Ponderation_Score'])}")
print(f"   - Mappings_XGBoost_Features : {len(data['Mappings_XGBoost_Features'])}")
print(f"   - Changelog_Mapping : {len(data['Changelog_Mapping'])}")

# ============================================================
# 9. AFFICHAGE DES RÈGLES
# ============================================================

print("\n📋 RÈGLES R001–R008 :")
if data['Regles']:
    for regle in data['Regles']:
        regle_id = regle.get('REGLE_ID', 'N/A')
        expression = regle.get('EXPRESSION_SQL', 'N/A')[:60] + "..." if len(str(regle.get('EXPRESSION_SQL', ''))) > 60 else regle.get('EXPRESSION_SQL', 'N/A')
        print(f"   - {regle_id} : {expression}")
else:
    print("   ⚠️ Aucune règle trouvée")

# ============================================================
# 10. AFFICHAGE DU CHANGELOG
# ============================================================

print("\n📋 DERNIÈRES MODIFICATIONS (Changelog) :")
if data['Changelog_Mapping']:
    for item in data['Changelog_Mapping'][-3:]:
        feature = item.get('Feature', 'N/A')
        v1 = item.get('Version_1_initiale', 'N/A')[:40] + "..." if len(str(item.get('Version_1_initiale', ''))) > 40 else item.get('Version_1_initiale', 'N/A')
        v2 = item.get('Version_2_corrigee', 'N/A')[:40] + "..." if len(str(item.get('Version_2_corrigee', ''))) > 40 else item.get('Version_2_corrigee', 'N/A')
        print(f"   - {feature} : {v2}")
else:
    print("   ⚠️ Aucune entrée dans le Changelog")

print("\n" + "=" * 60)
print("🏁 FIN - Référentiel JSON généré avec succès !")
print("=" * 60)

🚀 RÉGÉNÉRATION DU RÉFÉRENTIEL JSON (SANS NaN)

📁 Fichier Excel trouvé : ../metadata/mon_referentiel_final.xlsx

📋 Chargement des onglets...
   ✅ Modele_donnees : 16 lignes
   ✅ Objets_metier : 3 lignes
   ✅ Mappings : 5 lignes
   ✅ Contraintes : 10 lignes
   ✅ Regles : 8 lignes
   ✅ Valeurs_reference : 8 lignes
   ✅ Ponderation_Score : 5 lignes
   ✅ Mappings_XGBoost_Features : 9 lignes
   ✅ Changelog_Mapping : 7 lignes

🔍 Vérification des règles corrigées...
   ✅ R007 corrigé : SELECT VIN, COUNT(*) FROM FLEETVEHICLE GROUP BY VIN HAVING COUNT(*) > 1...

📦 Construction du JSON...

💾 Sauvegarde du JSON : ../metadata/referentiel.json
   ✅ Sauvegarde réussie

🔍 Vérification du JSON généré...
   ⚠️ Attention : des NaN sont encore présents dans le JSON
   Cela peut indiquer un problème de nettoyage
   📊 Taille du fichier : 13195 octets

📊 RÉSUMÉ DU RÉFÉRENTIEL
   Version : 2.1
   Date de mise à jour : 2026-08-11T12:25:41.879266

   Nombre d'entités :
   - Modele_donnees : 16
   - Objets_metie

In [2]:
# ============================================================
# UPDATE_JSON_V2.py
# Nettoyage renforcé des NaN
# ============================================================

import os
import pandas as pd
import json
import numpy as np
import math
from datetime import datetime

print("=" * 60)
print("🚀 RÉGÉNÉRATION JSON (NETTOYAGE RENFORCÉ)")
print("=" * 60)

# Chemins
CHEMIN_EXCEL = "../metadata/mon_referentiel_final.xlsx"
CHEMIN_JSON = "../metadata/referentiel.json"

# Vérifier l'existence du fichier
if not os.path.exists(CHEMIN_EXCEL):
    print(f"❌ Fichier non trouvé : {CHEMIN_EXCEL}")
    exit(1)

print(f"📁 Fichier trouvé : {CHEMIN_EXCEL}")

# Fonction de nettoyage robuste
def clean_value(v):
    """Nettoie une valeur individuelle pour éviter NaN"""
    if pd.isna(v):
        return None
    if isinstance(v, float) and math.isnan(v):
        return None
    if isinstance(v, str):
        v = v.strip()
        if v.lower() == 'nan' or v.lower() == 'none':
            return None
    return v

def clean_df(df):
    """Nettoie un DataFrame entier"""
    df_clean = df.copy()
    for col in df_clean.columns:
        df_clean[col] = df_clean[col].apply(clean_value)
    return df_clean

# Charger et nettoyer tous les onglets
print("\n📋 Chargement et nettoyage des onglets...")

df_modele = clean_df(pd.read_excel(CHEMIN_EXCEL, sheet_name="Modele_donnees"))
df_objets = clean_df(pd.read_excel(CHEMIN_EXCEL, sheet_name="Objets_metier"))
df_mappings = clean_df(pd.read_excel(CHEMIN_EXCEL, sheet_name="Mappings"))
df_contraintes = clean_df(pd.read_excel(CHEMIN_EXCEL, sheet_name="Contraintes"))
df_regles = clean_df(pd.read_excel(CHEMIN_EXCEL, sheet_name="Regles"))
df_valeurs = clean_df(pd.read_excel(CHEMIN_EXCEL, sheet_name="Valeurs_reference"))
df_ponderation = clean_df(pd.read_excel(CHEMIN_EXCEL, sheet_name="Ponderation_Score"))
df_mappings_xgboost = clean_df(pd.read_excel(CHEMIN_EXCEL, sheet_name="Mappings_XGBoost_Features"))
df_changelog = clean_df(pd.read_excel(CHEMIN_EXCEL, sheet_name="Changelog_Mapping"))

print("   ✅ Tous les onglets ont été nettoyés")

# Construire le JSON
data = {
    "version": "2.1",
    "date_maj": datetime.now().isoformat(),
    "Modele_donnees": df_modele.to_dict(orient='records'),
    "Objets_metier": df_objets.to_dict(orient='records'),
    "Mappings": df_mappings.to_dict(orient='records'),
    "Contraintes": df_contraintes.to_dict(orient='records'),
    "Regles": df_regles.to_dict(orient='records'),
    "Valeurs_reference": df_valeurs.to_dict(orient='records'),
    "Ponderation_Score": df_ponderation.to_dict(orient='records'),
    "Mappings_XGBoost_Features": df_mappings_xgboost.to_dict(orient='records'),
    "Changelog_Mapping": df_changelog.to_dict(orient='records')
}

# Sauvegarder
with open(CHEMIN_JSON, 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"\n💾 JSON sauvegardé : {CHEMIN_JSON}")

# Vérification : vérifier qu'il n'y a plus de NaN
with open(CHEMIN_JSON, 'r', encoding='utf-8') as f:
    contenu = f.read()

if 'NaN' in contenu:
    print("   ❌ Des NaN sont encore présents dans le JSON")
    print("   ⚠️ Vérifiez manuellement les cellules vides dans Excel")
else:
    print("   ✅ Aucun NaN détecté dans le JSON")

# Statistiques
print(f"\n📊 Taille du fichier : {os.path.getsize(CHEMIN_JSON)} octets")
print(f"   Version : {data['version']}")
print(f"   Date : {data['date_maj']}")

print("\n" + "=" * 60)
print("🏁 FIN")
print("=" * 60)

🚀 RÉGÉNÉRATION JSON (NETTOYAGE RENFORCÉ)
📁 Fichier trouvé : ../metadata/mon_referentiel_final.xlsx

📋 Chargement et nettoyage des onglets...
   ✅ Tous les onglets ont été nettoyés

💾 JSON sauvegardé : ../metadata/referentiel.json
   ❌ Des NaN sont encore présents dans le JSON
   ⚠️ Vérifiez manuellement les cellules vides dans Excel

📊 Taille du fichier : 13192 octets
   Version : 2.1
   Date : 2026-08-11T12:28:04.529012

🏁 FIN


In [3]:
# Cellule : Localiser les NaN dans le référentiel

import pandas as pd
import numpy as np

CHEMIN_EXCEL = "../metadata/mon_referentiel_final.xlsx"

# Fonction pour vérifier les NaN
def verifier_nan(df, nom):
    if df is None or df.empty:
        return
    for col in df.columns:
        if df[col].isna().any():
            nb_nan = df[col].isna().sum()
            print(f"   ⚠️ {nom}.{col} : {nb_nan} valeurs NaN")

print("🔍 Recherche des NaN dans chaque onglet...")
print("=" * 50)

# Charger et vérifier chaque onglet
df_modele = pd.read_excel(CHEMIN_EXCEL, sheet_name="Modele_donnees")
df_objets = pd.read_excel(CHEMIN_EXCEL, sheet_name="Objets_metier")
df_mappings = pd.read_excel(CHEMIN_EXCEL, sheet_name="Mappings")
df_contraintes = pd.read_excel(CHEMIN_EXCEL, sheet_name="Contraintes")
df_regles = pd.read_excel(CHEMIN_EXCEL, sheet_name="Regles")
df_valeurs = pd.read_excel(CHEMIN_EXCEL, sheet_name="Valeurs_reference")
df_ponderation = pd.read_excel(CHEMIN_EXCEL, sheet_name="Ponderation_Score")
df_mappings_xgboost = pd.read_excel(CHEMIN_EXCEL, sheet_name="Mappings_XGBoost_Features")
df_changelog = pd.read_excel(CHEMIN_EXCEL, sheet_name="Changelog_Mapping")

print("\n📋 NaN détectés :")
verifier_nan(df_modele, "Modele_donnees")
verifier_nan(df_objets, "Objets_metier")
verifier_nan(df_mappings, "Mappings")
verifier_nan(df_contraintes, "Contraintes")
verifier_nan(df_regles, "Regles")
verifier_nan(df_valeurs, "Valeurs_reference")
verifier_nan(df_ponderation, "Ponderation_Score")
verifier_nan(df_mappings_xgboost, "Mappings_XGBoost_Features")
verifier_nan(df_changelog, "Changelog_Mapping")

print("\n✅ Si aucun résultat n'apparaît, les NaN viennent d'ailleurs.")

🔍 Recherche des NaN dans chaque onglet...

📋 NaN détectés :
   ⚠️ Contraintes.VALEUR  : 7 valeurs NaN

✅ Si aucun résultat n'apparaît, les NaN viennent d'ailleurs.


In [4]:
# Cellule : Nettoyage spécifique de la colonne VALEUR

import os
import pandas as pd
import json
import numpy as np
from datetime import datetime

CHEMIN_EXCEL = "../metadata/mon_referentiel_final.xlsx"
CHEMIN_JSON = "../metadata/referentiel.json"

print("=" * 60)
print("🚀 NETTOYAGE CIBLÉ : Contraintes.VALEUR")
print("=" * 60)

# 1. Charger toutes les feuilles
print("\n📋 Chargement des onglets...")
df_modele = pd.read_excel(CHEMIN_EXCEL, sheet_name="Modele_donnees")
df_objets = pd.read_excel(CHEMIN_EXCEL, sheet_name="Objets_metier")
df_mappings = pd.read_excel(CHEMIN_EXCEL, sheet_name="Mappings")
df_contraintes = pd.read_excel(CHEMIN_EXCEL, sheet_name="Contraintes")
df_regles = pd.read_excel(CHEMIN_EXCEL, sheet_name="Regles")
df_valeurs = pd.read_excel(CHEMIN_EXCEL, sheet_name="Valeurs_reference")
df_ponderation = pd.read_excel(CHEMIN_EXCEL, sheet_name="Ponderation_Score")
df_mappings_xgboost = pd.read_excel(CHEMIN_EXCEL, sheet_name="Mappings_XGBoost_Features")
df_changelog = pd.read_excel(CHEMIN_EXCEL, sheet_name="Changelog_Mapping")

# 2. CORRECTION SPÉCIFIQUE : Nettoyer la colonne VALEUR
print("\n🔧 Nettoyage de la colonne VALEUR...")

# Compter les NaN avant
nb_nan_avant = df_contraintes['VALEUR'].isna().sum()
print(f"   - NaN avant : {nb_nan_avant}")

# Remplacer les NaN par None (sera converti en null en JSON)
df_contraintes['VALEUR'] = df_contraintes['VALEUR'].where(
    pd.notnull(df_contraintes['VALEUR']), 
    None
)

# Compter les NaN après
nb_nan_apres = df_contraintes['VALEUR'].isna().sum()
print(f"   - NaN après : {nb_nan_apres}")

# Vérifier le contenu de la colonne
print("\n📋 Aperçu des valeurs dans VALEUR :")
print(df_contraintes[['TABLE_NAME', 'COLUMN_NAME', 'TYPE_CONTRAINTE', 'VALEUR']].to_string(index=False))

# 3. Nettoyer les autres feuilles (au cas où)
print("\n🧹 Nettoyage général des autres feuilles...")

def clean_df(df):
    """Remplace tous les NaN par None"""
    return df.where(pd.notnull(df), None)

df_modele = clean_df(df_modele)
df_objets = clean_df(df_objets)
df_mappings = clean_df(df_mappings)
# df_contraintes déjà traité
df_regles = clean_df(df_regles)
df_valeurs = clean_df(df_valeurs)
df_ponderation = clean_df(df_ponderation)
df_mappings_xgboost = clean_df(df_mappings_xgboost)
df_changelog = clean_df(df_changelog)

# 4. Construire le JSON
print("\n📦 Construction du JSON...")
data = {
    "version": "2.1",
    "date_maj": datetime.now().isoformat(),
    "Modele_donnees": df_modele.to_dict(orient='records'),
    "Objets_metier": df_objets.to_dict(orient='records'),
    "Mappings": df_mappings.to_dict(orient='records'),
    "Contraintes": df_contraintes.to_dict(orient='records'),
    "Regles": df_regles.to_dict(orient='records'),
    "Valeurs_reference": df_valeurs.to_dict(orient='records'),
    "Ponderation_Score": df_ponderation.to_dict(orient='records'),
    "Mappings_XGBoost_Features": df_mappings_xgboost.to_dict(orient='records'),
    "Changelog_Mapping": df_changelog.to_dict(orient='records')
}

# 5. Sauvegarder
with open(CHEMIN_JSON, 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"\n💾 JSON sauvegardé : {CHEMIN_JSON}")

# 6. Vérification finale
with open(CHEMIN_JSON, 'r', encoding='utf-8') as f:
    contenu = f.read()

if 'NaN' in contenu:
    print("\n❌ Des NaN sont encore présents dans le JSON")
    print("   Recherche des NaN restants...")
    # Identifier où se trouvent les NaN
    import re
    for line in contenu.split('\n'):
        if 'NaN' in line:
            print(f"      {line[:100]}...")
else:
    print("\n✅ Aucun NaN détecté dans le JSON")

print(f"\n📊 Taille du fichier : {os.path.getsize(CHEMIN_JSON)} octets")
print("=" * 60)
print("🏁 FIN")
print("=" * 60)

🚀 NETTOYAGE CIBLÉ : Contraintes.VALEUR

📋 Chargement des onglets...

🔧 Nettoyage de la colonne VALEUR...


KeyError: 'VALEUR'

In [5]:
# Cellule : Vérifier les colonnes de la feuille Contraintes

import pandas as pd

CHEMIN_EXCEL = "../metadata/mon_referentiel_final.xlsx"

# Lire la feuille Contraintes
df_contraintes = pd.read_excel(CHEMIN_EXCEL, sheet_name="Contraintes")

# Afficher les noms de colonnes exacts
print("📋 Colonnes de la feuille Contraintes :")
print(df_contraintes.columns.tolist())

# Afficher les 5 premières lignes
print("\n📋 Aperçu des données :")
print(df_contraintes.head())

📋 Colonnes de la feuille Contraintes :
['TABLE_NAME', 'COLUMN_NAME', 'TYPE_CONTRAINTE ', 'VALEUR ']

📋 Aperçu des données :
  TABLE_NAME COLUMN_NAME TYPE_CONTRAINTE  VALEUR 
0   CONTRACT   REFERENCE         NOT_NULL     NaN
1   CONTRACT  START_DATE         NOT_NULL     NaN
2   CUSTOMER  FIRST_NAME         NOT_NULL     NaN
3   CUSTOMER   LAST_NAME         NOT_NULL     NaN
4   CUSTOMER       EMAIL           FORMAT   email


In [6]:
# Cellule : Nettoyage complet avec gestion des espaces

import os
import pandas as pd
import json
from datetime import datetime

CHEMIN_EXCEL = "../metadata/mon_referentiel_final.xlsx"
CHEMIN_JSON = "../metadata/referentiel.json"

print("=" * 60)
print("🚀 NETTOYAGE COMPLET (GESTION DES ESPACES)")
print("=" * 60)

# 1. Fonction pour nettoyer les noms de colonnes
def clean_column_names(df):
    """Supprime les espaces en début et fin des noms de colonnes"""
    df.columns = df.columns.str.strip()
    return df

# 2. Charger les feuilles avec nettoyage des noms de colonnes
print("\n📋 Chargement des onglets...")

df_modele = clean_column_names(pd.read_excel(CHEMIN_EXCEL, sheet_name="Modele_donnees"))
df_objets = clean_column_names(pd.read_excel(CHEMIN_EXCEL, sheet_name="Objets_metier"))
df_mappings = clean_column_names(pd.read_excel(CHEMIN_EXCEL, sheet_name="Mappings"))
df_contraintes = clean_column_names(pd.read_excel(CHEMIN_EXCEL, sheet_name="Contraintes"))
df_regles = clean_column_names(pd.read_excel(CHEMIN_EXCEL, sheet_name="Regles"))
df_valeurs = clean_column_names(pd.read_excel(CHEMIN_EXCEL, sheet_name="Valeurs_reference"))
df_ponderation = clean_column_names(pd.read_excel(CHEMIN_EXCEL, sheet_name="Ponderation_Score"))
df_mappings_xgboost = clean_column_names(pd.read_excel(CHEMIN_EXCEL, sheet_name="Mappings_XGBoost_Features"))
df_changelog = clean_column_names(pd.read_excel(CHEMIN_EXCEL, sheet_name="Changelog_Mapping"))

print("   ✅ Tous les onglets chargés avec noms de colonnes nettoyés")

# 3. Vérifier que la colonne VALEUR existe maintenant
print("\n🔍 Vérification des colonnes dans Contraintes :")
print(f"   Colonnes : {df_contraintes.columns.tolist()}")

if 'VALEUR' in df_contraintes.columns:
    print("   ✅ Colonne 'VALEUR' trouvée")
else:
    print("   ❌ Colonne 'VALEUR' toujours absente")
    print("   Arrêt du script.")
    exit()

# 4. Nettoyer la colonne VALEUR
print("\n🔧 Nettoyage de la colonne VALEUR...")

nb_nan_avant = df_contraintes['VALEUR'].isna().sum()
print(f"   - NaN avant : {nb_nan_avant}")

# Remplacer les NaN par None
df_contraintes['VALEUR'] = df_contraintes['VALEUR'].where(
    pd.notnull(df_contraintes['VALEUR']), 
    None
)

nb_nan_apres = df_contraintes['VALEUR'].isna().sum()
print(f"   - NaN après : {nb_nan_apres}")

# 5. Nettoyer toutes les autres colonnes
print("\n🧹 Nettoyage général de tous les DataFrames...")

def clean_df(df):
    return df.where(pd.notnull(df), None)

df_modele = clean_df(df_modele)
df_objets = clean_df(df_objets)
df_mappings = clean_df(df_mappings)
# df_contraintes déjà traité
df_regles = clean_df(df_regles)
df_valeurs = clean_df(df_valeurs)
df_ponderation = clean_df(df_ponderation)
df_mappings_xgboost = clean_df(df_mappings_xgboost)
df_changelog = clean_df(df_changelog)

# 6. Afficher un aperçu de la feuille Contraintes
print("\n📋 Aperçu des contraintes après nettoyage :")
print(df_contraintes[['TABLE_NAME', 'COLUMN_NAME', 'TYPE_CONTRAINTE', 'VALEUR']].to_string(index=False))

# 7. Construire le JSON
print("\n📦 Construction du JSON...")

data = {
    "version": "2.1",
    "date_maj": datetime.now().isoformat(),
    "Modele_donnees": df_modele.to_dict(orient='records'),
    "Objets_metier": df_objets.to_dict(orient='records'),
    "Mappings": df_mappings.to_dict(orient='records'),
    "Contraintes": df_contraintes.to_dict(orient='records'),
    "Regles": df_regles.to_dict(orient='records'),
    "Valeurs_reference": df_valeurs.to_dict(orient='records'),
    "Ponderation_Score": df_ponderation.to_dict(orient='records'),
    "Mappings_XGBoost_Features": df_mappings_xgboost.to_dict(orient='records'),
    "Changelog_Mapping": df_changelog.to_dict(orient='records')
}

# 8. Sauvegarder
with open(CHEMIN_JSON, 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"\n💾 JSON sauvegardé : {CHEMIN_JSON}")

# 9. Vérification finale
with open(CHEMIN_JSON, 'r', encoding='utf-8') as f:
    contenu = f.read()

if 'NaN' in contenu:
    print("\n❌ Des NaN sont encore présents dans le JSON")
    # Compter combien de fois NaN apparaît
    nb_nan = contenu.count('NaN')
    print(f"   Nombre d'occurrences : {nb_nan}")
else:
    print("\n✅ Aucun NaN détecté dans le JSON")

print(f"\n📊 Taille du fichier : {os.path.getsize(CHEMIN_JSON)} octets")
print("=" * 60)
print("🏁 FIN - JSON généré avec succès !")
print("=" * 60)

🚀 NETTOYAGE COMPLET (GESTION DES ESPACES)

📋 Chargement des onglets...
   ✅ Tous les onglets chargés avec noms de colonnes nettoyés

🔍 Vérification des colonnes dans Contraintes :
   Colonnes : ['TABLE_NAME', 'COLUMN_NAME', 'TYPE_CONTRAINTE', 'VALEUR']
   ✅ Colonne 'VALEUR' trouvée

🔧 Nettoyage de la colonne VALEUR...
   - NaN avant : 7
   - NaN après : 7

🧹 Nettoyage général de tous les DataFrames...

📋 Aperçu des contraintes après nettoyage :
  TABLE_NAME   COLUMN_NAME TYPE_CONTRAINTE         VALEUR
    CONTRACT     REFERENCE        NOT_NULL            NaN
    CONTRACT    START_DATE        NOT_NULL            NaN
    CUSTOMER    FIRST_NAME        NOT_NULL            NaN
    CUSTOMER     LAST_NAME        NOT_NULL            NaN
    CUSTOMER         EMAIL          FORMAT          email
FLEETVEHICLE LICENSE_PLATE        NOT_NULL            NaN
FLEETVEHICLE           VIN          UNIQUE            NaN
    CONTRACT        AMOUNT        POSITIVE            NaN
    CUSTOMER         PHONE   

In [7]:
# Cellule : CORRECTION FINALE DES NaN

import os
import pandas as pd
import json
import numpy as np
from datetime import datetime

CHEMIN_EXCEL = "../metadata/mon_referentiel_final.xlsx"
CHEMIN_JSON = "../metadata/referentiel.json"

print("=" * 60)
print("🚀 CORRECTION FINALE DES NaN")
print("=" * 60)

# 1. Charger toutes les feuilles
print("\n📋 Chargement des onglets...")

def clean_cols(df):
    df.columns = df.columns.str.strip()
    return df

df_modele = clean_cols(pd.read_excel(CHEMIN_EXCEL, sheet_name="Modele_donnees"))
df_objets = clean_cols(pd.read_excel(CHEMIN_EXCEL, sheet_name="Objets_metier"))
df_mappings = clean_cols(pd.read_excel(CHEMIN_EXCEL, sheet_name="Mappings"))
df_contraintes = clean_cols(pd.read_excel(CHEMIN_EXCEL, sheet_name="Contraintes"))
df_regles = clean_cols(pd.read_excel(CHEMIN_EXCEL, sheet_name="Regles"))
df_valeurs = clean_cols(pd.read_excel(CHEMIN_EXCEL, sheet_name="Valeurs_reference"))
df_ponderation = clean_cols(pd.read_excel(CHEMIN_EXCEL, sheet_name="Ponderation_Score"))
df_mappings_xgboost = clean_cols(pd.read_excel(CHEMIN_EXCEL, sheet_name="Mappings_XGBoost_Features"))
df_changelog = clean_cols(pd.read_excel(CHEMIN_EXCEL, sheet_name="Changelog_Mapping"))

print("   ✅ Tous les onglets chargés")

# 2. Fonction qui convertit un DataFrame en liste de dict en remplaçant les NaN par None
def df_to_clean_records(df):
    records = []
    for _, row in df.iterrows():
        record = {}
        for col in df.columns:
            val = row[col]
            # Vérifier si c'est NaN ou NaT
            if pd.isna(val):
                record[col] = None
            elif isinstance(val, float) and np.isnan(val):
                record[col] = None
            else:
                record[col] = val
        records.append(record)
    return records

# 3. Convertir toutes les feuilles
print("\n🔄 Conversion des données...")

data = {
    "version": "2.1",
    "date_maj": datetime.now().isoformat(),
    "Modele_donnees": df_to_clean_records(df_modele),
    "Objets_metier": df_to_clean_records(df_objets),
    "Mappings": df_to_clean_records(df_mappings),
    "Contraintes": df_to_clean_records(df_contraintes),
    "Regles": df_to_clean_records(df_regles),
    "Valeurs_reference": df_to_clean_records(df_valeurs),
    "Ponderation_Score": df_to_clean_records(df_ponderation),
    "Mappings_XGBoost_Features": df_to_clean_records(df_mappings_xgboost),
    "Changelog_Mapping": df_to_clean_records(df_changelog)
}

print("   ✅ Conversion terminée")

# 4. Vérifier qu'il n'y a plus de NaN dans les records
def check_nan_in_records(records, name):
    count = 0
    for record in records:
        for key, val in record.items():
            if isinstance(val, float) and np.isnan(val):
                count += 1
            elif isinstance(val, str) and val.lower() == 'nan':
                count += 1
    if count > 0:
        print(f"   ⚠️ {name} : {count} NaN résiduels")
    return count

print("\n🔍 Vérification des NaN résiduels...")
total_nan = 0
for key, records in data.items():
    if key not in ['version', 'date_maj']:
        total_nan += check_nan_in_records(records, key)

print(f"\n   Total NaN trouvés : {total_nan}")

if total_nan == 0:
    print("   ✅ Aucun NaN détecté !")
else:
    print("   ⚠️ Des NaN persistent, on va les forcer manuellement")

# 5. Sauvegarder le JSON
with open(CHEMIN_JSON, 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"\n💾 JSON sauvegardé : {CHEMIN_JSON}")

# 6. Vérification finale avec lecture du fichier
with open(CHEMIN_JSON, 'r', encoding='utf-8') as f:
    contenu = f.read()

if 'NaN' in contenu:
    print("\n❌ Le fichier JSON contient encore la chaîne 'NaN'")
    print("   Procédure de force brute : remplacement dans le fichier texte")

    # Force brute : remplacer les NaN dans le texte
    with open(CHEMIN_JSON, 'r', encoding='utf-8') as f:
        contenu = f.read()
    contenu = contenu.replace('NaN', 'null')
    with open(CHEMIN_JSON, 'w', encoding='utf-8') as f:
        f.write(contenu)
    print("   ✅ Remplacement forcé effectué")
else:
    print("\n✅ Aucun NaN détecté dans le JSON final")

print(f"\n📊 Taille du fichier : {os.path.getsize(CHEMIN_JSON)} octets")
print("=" * 60)
print("🏁 FIN - JSON corrigé avec succès !")
print("=" * 60)

🚀 CORRECTION FINALE DES NaN

📋 Chargement des onglets...
   ✅ Tous les onglets chargés

🔄 Conversion des données...
   ✅ Conversion terminée

🔍 Vérification des NaN résiduels...

   Total NaN trouvés : 0
   ✅ Aucun NaN détecté !

💾 JSON sauvegardé : ../metadata/referentiel.json

✅ Aucun NaN détecté dans le JSON final

📊 Taille du fichier : 13177 octets
🏁 FIN - JSON corrigé avec succès !


In [8]:
# Cellule : Validation de l'import du référentiel

import os
import json
from models import Referentiel

CHEMIN_JSON = "../metadata/referentiel.json"

print("=" * 60)
print("🧪 VALIDATION DE L'IMPORT DU RÉFÉRENTIEL")
print("=" * 60)

# 1. Vérifier que le fichier existe
if not os.path.exists(CHEMIN_JSON):
    print(f"❌ Fichier JSON non trouvé : {CHEMIN_JSON}")
    exit(1)

print(f"📁 Fichier JSON trouvé : {CHEMIN_JSON}")

# 2. Lire le fichier JSON
try:
    with open(CHEMIN_JSON, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print("✅ Structure JSON valide")
except Exception as e:
    print(f"❌ Erreur de parsing JSON : {e}")
    exit(1)

# 3. Vérifier les sections obligatoires
sections = ['Modele_donnees', 'Objets_metier', 'Mappings', 'Contraintes', 'Regles', 'Valeurs_reference', 'Ponderation_Score', 'Mappings_XGBoost_Features']

print("\n📋 Vérification des sections...")
for section in sections:
    if section in data:
        print(f"   ✅ {section} : {len(data[section])} éléments")
    else:
        print(f"   ❌ {section} manquant")

# 4. Vérifier R007
regles = data.get('Regles', [])
r007 = next((r for r in regles if r.get('REGLE_ID') == 'R007'), None)
if r007:
    expr = r007.get('EXPRESSION_SQL', '')
    if 'COUNT(*)' in expr:
        print(f"\n✅ R007 corrigé : {expr[:80]}...")
    else:
        print(f"\n⚠️ R007 : {expr[:80]}...")

# 5. Tester l'import avec Referentiel
print("\n🔄 Test d'import via Referentiel.from_json()...")
try:
    referentiel = Referentiel.from_json(CHEMIN_JSON)
    print("✅ Import réussi !")
    print(f"   - Tables : {len(referentiel.tables)}")
    print(f"   - Règles : {len(referentiel.regles)}")
    print(f"   - Contraintes : {len(referentiel.contraintes)}")
    print(f"   - Mappings XGBoost : {len(referentiel.mappings)}")
except Exception as e:
    print(f"❌ Erreur : {e}")

print("\n" + "=" * 60)
print("🏁 FIN DE LA VALIDATION")
print("=" * 60)

🧪 VALIDATION DE L'IMPORT DU RÉFÉRENTIEL
📁 Fichier JSON trouvé : ../metadata/referentiel.json
✅ Structure JSON valide

📋 Vérification des sections...
   ✅ Modele_donnees : 16 éléments
   ✅ Objets_metier : 3 éléments
   ✅ Mappings : 5 éléments
   ✅ Contraintes : 10 éléments
   ✅ Regles : 8 éléments
   ✅ Valeurs_reference : 8 éléments
   ✅ Ponderation_Score : 5 éléments
   ✅ Mappings_XGBoost_Features : 9 éléments

✅ R007 corrigé : SELECT VIN, COUNT(*) FROM FLEETVEHICLE GROUP BY VIN HAVING COUNT(*) > 1...

🔄 Test d'import via Referentiel.from_json()...
✅ Import réussi !
   - Tables : 0
   - Règles : 0
   - Contraintes : 0
   - Mappings XGBoost : 0

🏁 FIN DE LA VALIDATION


In [9]:
# Cellule : Re-valider l'import du référentiel

import os
import json
from models import Referentiel

CHEMIN_JSON = "../metadata/referentiel.json"

print("=" * 60)
print("🧪 RE-VALIDATION DE L'IMPORT DU RÉFÉRENTIEL")
print("=" * 60)

try:
    referentiel = Referentiel.from_json(CHEMIN_JSON)
    print("✅ Import réussi !")
    print(f"   - Tables : {len(referentiel.tables)}")
    print(f"   - Règles : {len(referentiel.regles)}")
    print(f"   - Contraintes : {len(referentiel.contraintes)}")
    print(f"   - Mappings XGBoost : {len(referentiel.mappings_xgboost)}")
    
    # Afficher les règles
    print("\n📋 Règles chargées :")
    for regle in referentiel.regles:
        print(f"   - {regle.id} : {regle.nom}")
    
    # Afficher les mappings XGBoost
    print("\n📋 Mappings XGBoost :")
    for mapping in referentiel.mappings_xgboost:
        print(f"   - {mapping['feature']} → {mapping['table_miles']}.{mapping['colonne_miles']}")
    
except Exception as e:
    print(f"❌ Erreur : {e}")

🧪 RE-VALIDATION DE L'IMPORT DU RÉFÉRENTIEL
✅ Import réussi !
   - Tables : 0
   - Règles : 0
   - Contraintes : 0
❌ Erreur : 'Referentiel' object has no attribute 'mappings_xgboost'


In [10]:
# Cellule : Re-valider après correction

import os
import json
from models import Referentiel

CHEMIN_JSON = "../metadata/referentiel.json"

print("=" * 60)
print("🧪 RE-VALIDATION FINALE")
print("=" * 60)

try:
    referentiel = Referentiel.from_json(CHEMIN_JSON)
    print("✅ Import réussi !")
    print(f"   - Tables : {len(referentiel.tables)}")
    print(f"   - Règles : {len(referentiel.regles)}")
    print(f"   - Contraintes : {len(referentiel.contraintes)}")
    print(f"   - Mappings XGBoost : {len(referentiel.mappings_xgboost)}")
    
    print("\n📋 Règles chargées :")
    for regle in referentiel.regles:
        print(f"   - {regle.id} : {regle.nom}")
    
    print("\n📋 Mappings XGBoost :")
    for mapping in referentiel.mappings_xgboost:
        print(f"   - {mapping['feature']} → {mapping['table_miles']}.{mapping['colonne_miles']}")
    
except Exception as e:
    print(f"❌ Erreur : {e}")

🧪 RE-VALIDATION FINALE
✅ Import réussi !
   - Tables : 0
   - Règles : 0
   - Contraintes : 0
❌ Erreur : 'Referentiel' object has no attribute 'mappings_xgboost'


In [1]:
# Cellule : Recharger le module models.py

import importlib
import models
importlib.reload(models)
print("✅ Module models rechargé !")

✅ Module models rechargé !


In [2]:
# Cellule : Re-valider après correction

import os
import json
from models import Referentiel

CHEMIN_JSON = "../metadata/referentiel.json"

print("=" * 60)
print("🧪 RE-VALIDATION FINALE")
print("=" * 60)

try:
    referentiel = Referentiel.from_json(CHEMIN_JSON)
    print("✅ Import réussi !")
    print(f"   - Tables : {len(referentiel.tables)}")
    print(f"   - Règles : {len(referentiel.regles)}")
    print(f"   - Contraintes : {len(referentiel.contraintes)}")
    print(f"   - Mappings XGBoost : {len(referentiel.mappings_xgboost)}")
    
    print("\n📋 Règles chargées :")
    for regle in referentiel.regles:
        print(f"   - {regle.id} : {regle.nom}")
    
    print("\n📋 Mappings XGBoost :")
    for mapping in referentiel.mappings_xgboost:
        print(f"   - {mapping['feature']} → {mapping['table_miles']}.{mapping['colonne_miles']}")
    
except Exception as e:
    print(f"❌ Erreur : {e}")

🧪 RE-VALIDATION FINALE
✅ Import réussi !
   - Tables : 3
   - Règles : 8
   - Contraintes : 10
   - Mappings XGBoost : 9

📋 Règles chargées :
   - R001 : Contrat_Reference_Non_Nulle
   - R002 : Client_Prenom_Non_Nul
   - R003 : Client_Email_Valide
   - R004 : Plaque_Immatriculation_Non_Nulle
   - R005 : Montant_Contrat_Positif
   - R006 : Date_Fin_Apres_Debut
   - R007 : VIN_Unique
   - R008 : Client_Nom_Complet_Non_Nul

📋 Mappings XGBoost :
   - amount → CREDITSCORE.REQUESTEDAMOUNT
   - amount_autorise → CREDITSCORE.AUTHORIZEDAMOUNT
   - duration → LEASESERVICE.DURATION
   - interest_rate → LEASESERVICE.INTERESTRATE
   - anciennete_client → BUSINESSPARTNER.ESTABLISHMENTDATE (derive)
   - credit_score → CREDITSCORE.NUMERICSCORE
   - risk_percentage → CREDITSCORE.RISKPERCENTAGE
   - anciennete_contrat → CONTRACT.STARTDATE (derive)
   - is_anomaly → (dérivé des règles).(N/A)


In [3]:
# Cellule : Tester toutes les règles SQL R001–R008

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

# Créer des données de test pour chaque table
n = 100

# CONTRACT
contract_data = {
    "CONTRACT_ID": range(1, n+1),
    "REFERENCE": [f"REF{i:03d}" for i in range(1, n+1)],
    "START_DATE": [datetime(2024, 1, 1) + timedelta(days=i*10) for i in range(n)],
    "END_DATE": [datetime(2024, 12, 31) - timedelta(days=i*5) for i in range(n)],
    "AMOUNT": np.random.normal(15000, 5000, n)
}
# Ajouter des anomalies pour tester les règles
contract_data["REFERENCE"][10] = None  # R001 : NULL
contract_data["START_DATE"][20] = None  # R001 : NULL
contract_data["END_DATE"][30] = datetime(2023, 12, 31)  # R006 : < START_DATE
contract_data["AMOUNT"][40] = -5000  # R005 : négatif

df_contract = pd.DataFrame(contract_data)

# CUSTOMER
customer_data = {
    "CUSTOMER_ID": range(1, n+1),
    "FIRST_NAME": [f"First{i}" for i in range(1, n+1)],
    "LAST_NAME": [f"Last{i}" for i in range(1, n+1)],
    "EMAIL": [f"email{i}@test.com" for i in range(1, n+1)],
    "PHONE": [f"0{i:09d}" for i in range(1, n+1)]
}
# Ajouter des anomalies
customer_data["FIRST_NAME"][15] = None  # R002 : NULL
customer_data["LAST_NAME"][25] = None  # R008 : NULL
customer_data["EMAIL"][35] = "invalid_email"  # R003 : format invalide
customer_data["PHONE"][45] = "12345"  # R003 : format invalide (trop court)

df_customer = pd.DataFrame(customer_data)

# FLEETVEHICLE
fleet_data = {
    "FLEETVEHICLE_ID": range(1, n+1),
    "LICENSE_PLATE": [f"AB-{i:03d}-CD" for i in range(1, n+1)],
    "VIN": [f"VIN{i:05d}" for i in range(1, n+1)],
    "MAKE": ["Renault"] * n,
    "MODEL": ["Clio"] * n,
    "YEAR": np.random.randint(2018, 2025, n)
}
# Ajouter des anomalies
fleet_data["LICENSE_PLATE"][50] = None  # R004 : NULL
fleet_data["VIN"][55] = "VIN00001"  # R007 : doublon (même VIN)
fleet_data["VIN"][56] = "VIN00001"  # R007 : doublon

df_fleet = pd.DataFrame(fleet_data)

print("📊 Données de test créées :")
print(f"   - CONTRACT : {len(df_contract)} lignes")
print(f"   - CUSTOMER : {len(df_customer)} lignes")
print(f"   - FLEETVEHICLE : {len(df_fleet)} lignes")

# Exécuter les règles
print("\n" + "=" * 60)
print("🧪 EXÉCUTION DES RÈGLES R001–R008")
print("=" * 60)

regles = [
    {
        "id": "R001",
        "nom": "Contrat_Reference_Non_Nulle",
        "sql": "SELECT * FROM CONTRACT WHERE REFERENCE IS NULL",
        "df": df_contract
    },
    {
        "id": "R002",
        "nom": "Client_Prenom_Non_Nul",
        "sql": "SELECT * FROM CUSTOMER WHERE FIRST_NAME IS NULL",
        "df": df_customer
    },
    {
        "id": "R003",
        "nom": "Client_Email_Valide",
        "sql": "SELECT * FROM CUSTOMER WHERE EMAIL NOT LIKE '%@%.%'",
        "df": df_customer
    },
    {
        "id": "R004",
        "nom": "Plaque_Immatriculation_Non_Nulle",
        "sql": "SELECT * FROM FLEETVEHICLE WHERE LICENSE_PLATE IS NULL",
        "df": df_fleet
    },
    {
        "id": "R005",
        "nom": "Montant_Contrat_Positif",
        "sql": "SELECT * FROM CONTRACT WHERE AMOUNT < 0",
        "df": df_contract
    },
    {
        "id": "R006",
        "nom": "Date_Fin_Apres_Debut",
        "sql": "SELECT * FROM CONTRACT WHERE END_DATE < START_DATE",
        "df": df_contract
    },
    {
        "id": "R007",
        "nom": "VIN_Unique",
        "sql": "SELECT VIN, COUNT(*) FROM FLEETVEHICLE GROUP BY VIN HAVING COUNT(*) > 1",
        "df": df_fleet
    },
    {
        "id": "R008",
        "nom": "Client_Nom_Complet_Non_Nul",
        "sql": "SELECT * FROM CUSTOMER WHERE FIRST_NAME IS NULL OR LAST_NAME IS NULL",
        "df": df_customer
    }
]

resultats = []
for regle in regles:
    df = regle["df"]
    sql = regle["sql"].upper()
    
    # Simuler l'exécution SQL
    if "REFERENCE IS NULL" in sql:
        resultat = df[df["REFERENCE"].isna()]
    elif "FIRST_NAME IS NULL" in sql and "OR LAST_NAME IS NULL" not in sql:
        resultat = df[df["FIRST_NAME"].isna()]
    elif "FIRST_NAME IS NULL OR LAST_NAME IS NULL" in sql:
        resultat = df[df["FIRST_NAME"].isna() | df["LAST_NAME"].isna()]
    elif "EMAIL NOT LIKE" in sql:
        resultat = df[~df["EMAIL"].astype(str).str.contains('@', na=False)]
    elif "LICENSE_PLATE IS NULL" in sql:
        resultat = df[df["LICENSE_PLATE"].isna()]
    elif "AMOUNT < 0" in sql:
        resultat = df[df["AMOUNT"] < 0]
    elif "END_DATE < START_DATE" in sql:
        resultat = df[df["END_DATE"] < df["START_DATE"]]
    elif "COUNT(*)" in sql and "GROUP BY VIN" in sql:
        doublons = df.groupby("VIN").size()
        doublons = doublons[doublons > 1]
        resultat = doublons
    else:
        resultat = pd.DataFrame()
    
    resultats.append({
        "id": regle["id"],
        "nom": regle["nom"],
        "nb_anomalies": len(resultat) if not isinstance(resultat, pd.Series) else len(resultat),
        "resultat": resultat
    })

# Afficher les résultats
print("\n📋 RÉSULTATS DES TESTS :")
print("-" * 50)

for r in resultats:
    if r["nb_anomalies"] > 0:
        print(f"   ❌ {r['id']} : {r['nom']} → {r['nb_anomalies']} anomalies")
    else:
        print(f"   ✅ {r['id']} : {r['nom']} → Aucune anomalie")

# Détail des anomalies
print("\n📋 DÉTAIL DES ANOMALIES :")
for r in resultats:
    if r["nb_anomalies"] > 0:
        print(f"\n   🔴 {r['id']} - {r['nom']} :")
        if isinstance(r["resultat"], pd.Series):
            print(f"      Doublons : {r['resultat'].to_dict()}")
        else:
            print(f"      {r['resultat'][['CONTRACT_ID' if 'CONTRACT_ID' in r['resultat'].columns else 'CUSTOMER_ID' if 'CUSTOMER_ID' in r['resultat'].columns else 'FLEETVEHICLE_ID']].to_string(index=False)}")

print("\n" + "=" * 60)
print("🏁 FIN DES TESTS")
print("=" * 60)

📊 Données de test créées :
   - CONTRACT : 100 lignes
   - CUSTOMER : 100 lignes
   - FLEETVEHICLE : 100 lignes

🧪 EXÉCUTION DES RÈGLES R001–R008

📋 RÉSULTATS DES TESTS :
--------------------------------------------------
   ❌ R001 : Contrat_Reference_Non_Nulle → 1 anomalies
   ❌ R002 : Client_Prenom_Non_Nul → 1 anomalies
   ❌ R003 : Client_Email_Valide → 1 anomalies
   ❌ R004 : Plaque_Immatriculation_Non_Nulle → 1 anomalies
   ❌ R005 : Montant_Contrat_Positif → 1 anomalies
   ❌ R006 : Date_Fin_Apres_Debut → 75 anomalies
   ❌ R007 : VIN_Unique → 1 anomalies
   ❌ R008 : Client_Nom_Complet_Non_Nul → 2 anomalies

📋 DÉTAIL DES ANOMALIES :

   🔴 R001 - Contrat_Reference_Non_Nulle :
       CONTRACT_ID
          11

   🔴 R002 - Client_Prenom_Non_Nul :
       CUSTOMER_ID
          16

   🔴 R003 - Client_Email_Valide :
       CUSTOMER_ID
          36

   🔴 R004 - Plaque_Immatriculation_Non_Nulle :
       FLEETVEHICLE_ID
              51

   🔴 R005 - Montant_Contrat_Positif :
       CONTRACT_ID
